In [1]:
import polars as pl
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold

In [2]:
print("--- 1. Initializing Lazy Loader & Strict Schema Parsing ---")
file_path = '../data/processed/train_data_fe_completed.parquet'
lf = pl.scan_parquet(file_path)

# Read the exact data types from the file
schema = pl.read_parquet_schema(file_path)

# 11 Categorical columns defined by Amex
amex_cats = ['B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68']

num_cols = []
cat_cols = []

# Dynamically sort columns based on their ACTUAL Polars data type
for col_name, dtype in schema.items():
    # Skip our keys and targets
    if col_name in ['customer_ID', 'S_2', 'target', 'anomaly_score']:
        continue
        
    # If Amex says it's categorical, OR Polars sees it as a String/Categorical
    if col_name in amex_cats or dtype in [pl.Categorical, pl.String]:
        cat_cols.append(col_name)
    # If Polars sees it as a number
    elif dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int64]:
        num_cols.append(col_name)

print(f"Verified Types: {len(num_cols)} numerical and {len(cat_cols)} categorical features.")

--- 1. Initializing Lazy Loader & Strict Schema Parsing ---
Verified Types: 161 numerical and 32 categorical features.


In [3]:
print("--- 2. Building Aggregation Engine ---")
exprs = []

# Numeric Aggregations (Guaranteed to only run on floats/ints)
for c in num_cols:
    exprs.extend([
        pl.col(c).mean().alias(f"{c}_mean"),
        pl.col(c).min().alias(f"{c}_min"),
        pl.col(c).max().alias(f"{c}_max"),
        pl.col(c).last().alias(f"{c}_last")
    ])

# Categorical Aggregations
for c in cat_cols:
    exprs.extend([
        pl.col(c).last().alias(f"{c}_last"),
        pl.col(c).n_unique().alias(f"{c}_nunique")
    ])

# Anomaly Aggregation
if 'anomaly_score' in schema.keys():
    exprs.extend([
        pl.col("anomaly_score").min().alias("anomaly_score_worst"),
        pl.col("anomaly_score").last().alias("anomaly_score_last")
    ])

--- 2. Building Aggregation Engine ---


In [4]:
print("--- 3. Executing Streaming Aggregation ---")
# Process the 5.5M rows
df_agg = lf.group_by('customer_ID').agg(exprs).collect(streaming=True)

# Bring the target back in
target_df = lf.group_by('customer_ID').agg(pl.col('target').first()).collect()
df_agg = df_agg.join(target_df, on='customer_ID', how='inner')

print(f" Aggregation complete! Shape: {df_agg.shape}")

--- 3. Executing Streaming Aggregation ---


C:\Users\santo\AppData\Local\Temp\ipykernel_26396\2295892956.py:3: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_agg = lf.group_by('customer_ID').agg(exprs).collect(streaming=True)


 Aggregation complete! Shape: (458913, 712)


In [5]:
print("--- 4. Memory Cleanup ---")
# Delete the Polars LazyFrame references and explicitly clear RAM
# before we attempt to convert this massive table to Pandas.
del lf
del target_df
gc.collect()

--- 4. Memory Cleanup ---


8

In [6]:
print("--- 5. Preparing for Leak-Proof Encoding ---")
# Convert to Pandas safely
df_final = df_agg.to_pandas()

del df_agg
gc.collect()

y = df_final['target'].astype(int)
X = df_final.drop(columns=['target', 'customer_ID'])

# Identify categorical features to encode (only the ones ending in _last)
cat_features_to_encode = [f"{c}_last" for c in cat_cols if f"{c}_last" in X.columns]

for col in cat_features_to_encode:
    X[f"{col}_target_enc"] = np.nan

print("--- 6. Target Encoding with Leakage Control ---")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    print(f"Processing Target Encoding for Fold {fold}/5...")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train = y.iloc[train_idx]
    
    for col in cat_features_to_encode:
        target_means = y_train.groupby(X_train[col], observed=False).mean()
        X.loc[val_idx, f"{col}_target_enc"] = X_val[col].map(target_means)
        
    fold += 1

global_mean = y.mean()
for col in cat_features_to_encode:
    X[f"{col}_target_enc"] = X[f"{col}_target_enc"].fillna(global_mean)
    
X = X.drop(columns=cat_features_to_encode)

print(" Leak-Proof Target Encoding Complete!")

--- 5. Preparing for Leak-Proof Encoding ---
--- 6. Target Encoding with Leakage Control ---
Processing Target Encoding for Fold 1/5...
Processing Target Encoding for Fold 2/5...
Processing Target Encoding for Fold 3/5...
Processing Target Encoding for Fold 4/5...
Processing Target Encoding for Fold 5/5...
 Leak-Proof Target Encoding Complete!


In [7]:
print("--- Target Encoding Leakage Check ---")

# Let's check how many unique values are in our encoded columns.
# We will look at columns that end in '_target_enc'
encoded_cols = [c for c in X.columns if c.endswith('_target_enc')]

for col in encoded_cols[:5]: # Just check the first 5 to keep it clean
    unique_values = X[col].nunique()
    print(f"{col}: {unique_values} unique encoded values")

--- Target Encoding Leakage Check ---
D_49_last_target_enc: 6 unique encoded values
D_63_last_target_enc: 30 unique encoded values
D_64_last_target_enc: 20 unique encoded values
D_66_last_target_enc: 13 unique encoded values
D_68_last_target_enc: 33 unique encoded values


In [8]:
print("--- 7. Saving Final ML Dataset ---")
X['customer_ID'] = df_final['customer_ID']
X['target'] = y

final_save_path = '../data/processed/train_data_ml_ready.parquet'
X.to_parquet(final_save_path)
print(f" Final dataset saved at: {final_save_path}")

--- 7. Saving Final ML Dataset ---
 Final dataset saved at: ../data/processed/train_data_ml_ready.parquet
